In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🚛 Scania APS Failure Prediction\n",
    "## 📊 Exploratory Data Analysis & Data Profiling\n",
    "\n",
    "**Author**: Data Scientist  \n",
    "**Date**: 2026  \n",
    "**Purpose**: Comprehensive data understanding to drive feature engineering and modeling decisions\n",
    "\n",
    "---\n",
    "\n",
    "## 📋 Table of Contents\n",
    "1. [Dataset Overview](#dataset-overview)\n",
    "2. [Class Distribution Analysis](#class-distribution-analysis)\n",
    "3. [Missing Value Analysis](#missing-value-analysis)\n",
    "4. [Feature Analysis](#feature-analysis)\n",
    "5. [Correlation Analysis](#correlation-analysis)\n",
    "6. [Business Insights](#business-insights)\n",
    "7. [Summary & Next Steps](#summary--next-steps)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import plotly.express as px\n",
    "import plotly.graph_objects as go\n",
    "from plotly.subplots import make_subplots\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set display options\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.max_rows', 100)\n",
    "pd.set_option('display.float_format', lambda x: '%.2f' % x)\n",
    "\n",
    "# Custom color palette\n",
    "COLORS = {\n",
    "    'primary': '#003366',   # Scania blue\n",
    "    'secondary': '#FFB612', # Scania yellow\n",
    "    'success': '#28A745',\n",
    "    'danger': '#DC3545',\n",
    "    'warning': '#FFC107',\n",
    "    'info': '#17A2B8'\n",
    "}"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📂 Dataset Overview\n",
    "\n",
    "### Loading Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load training and test data\n",
    "train_df = pd.read_csv('../data/raw/train.csv')\n",
    "test_df = pd.read_csv('../data/raw/test.csv')\n",
    "\n",
    "print(\"=\"*60)\n",
    "print(\"📊 DATASET INFORMATION\")\n",
    "print(\"=\"*60)\n",
    "print(f\"Training set shape: {train_df.shape}\")\n",
    "print(f\"Test set shape: {test_df.shape}\")\n",
    "print(f\"Total features: {train_df.shape[1] - 1}\")  # Excluding target"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display first few rows\n",
    "train_df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 📈 Key Statistics\n",
    "\n",
    "**Insights**:\n",
    "- The dataset contains 170+ anonymized features representing sensor readings\n",
    "- Features include counters, histogram bins, and various operational parameters\n",
    "- `class` is the target variable with values 'pos' (APS failure) and 'neg' (other failure)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Data types and memory usage\n",
    "memory_usage = train_df.memory_usage(deep=True).sum() / 1024**2\n",
    "\n",
    "print(\"📊 DATASET STATISTICS\")\n",
    "print(\"=\"*60)\n",
    "print(f\"Memory usage: {memory_usage:.2f} MB\")\n",
    "print(f\"Number of features: {train_df.shape[1] - 1}\")\n",
    "print(f\"Number of records: {train_df.shape[0]}\")\n",
    "\n",
    "# Data type distribution\n",
    "dtype_counts = train_df.dtypes.value_counts()\n",
    "print(\"\\n📊 Data Type Distribution:\")\n",
    "for dtype, count in dtype_counts.items():\n",
    "    print(f\"  {dtype}: {count} columns\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🎯 Class Distribution Analysis\n",
    "\n",
    "Understanding the target distribution is crucial for:\n",
    "1. Handling class imbalance\n",
    "2. Choosing appropriate evaluation metrics\n",
    "3. Setting business expectations"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Class distribution\n",
    "class_counts = train_df['class'].value_counts()\n",
    "class_percentages = (class_counts / len(train_df) * 100).round(2)\n",
    "\n",
    "# Create dataframe for visualization\n",
    "class_df = pd.DataFrame({\n",
    "    'Class': class_counts.index,\n",
    "    'Count': class_counts.values,\n",
    "    'Percentage': class_percentages.values\n",
    "})\n",
    "\n",
    "# Create interactive bar chart\n",
    "fig = px.bar(\n",
    "    class_df,\n",
    "    x='Class',\n",
    "    y='Count',\n",
    "    text='Percentage',\n",
    "    color='Class',\n",
    "    color_discrete_map={'neg': COLORS['primary'], 'pos': COLORS['secondary']},\n",
    "    title='Class Distribution in Training Set',\n",
    "    labels={'Count': 'Number of Samples', 'Class': 'Failure Type'}\n",
    ")\n",
    "\n",
    "fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')\n",
    "fig.update_layout(\n",
    "    height=500,\n",
    "    showlegend=False,\n",
    "    plot_bgcolor='white',\n",
    "    font=dict(size=14)\n",
    ")\n",
    "fig.show()\n",
    "\n",
    "# Print statistics\n",
    "print(\"📊 CLASS DISTRIBUTION\")\n",
    "print(\"=\"*60)\n",
    "for idx, row in class_df.iterrows():\n",
    "    print(f\"{row['Class'].upper()}: {row['Count']:,} samples ({row['Percentage']:.2f}%)\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "imbalance_ratio = class_counts['neg'] / class_counts['pos']\n",
    "print(f\"\\n⚠️ Imbalance Ratio: {imbalance_ratio:.2f}:1\")\n",
    "print(f\"   (Negative is {imbalance_ratio:.2f}x more frequent than Positive)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 💼 Business Insight: Class Imbalance\n",
    "\n",
    "**Key Observations**:\n",
    "1. **Significant class imbalance**: Negatives (other failures) heavily outnumber Positives (APS failures)\n",
    "2. **Real-world reflection**: APS failures are rare but costly - this is realistic\n",
    "3. **Modeling implications**: \n",
    "   - Standard accuracy will be misleading\n",
    "   - Need to use F1-score, precision, recall\n",
    "   - MUST use cost-sensitive learning\n",
    "   - Threshold must be adjusted for business costs\n",
    "\n",
    "**Business Impact**:\n",
    "- Each missed APS failure (FN) costs 500 units\n",
    "- Each false alarm (FP) costs 10 units\n",
    "- **We should optimize for minimizing FN, even at the cost of more FP**"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🔍 Missing Value Analysis\n",
    "\n",
    "The dataset contains `na` values which represent missing sensor readings. Understanding missing patterns is crucial for:\n",
    "1. Choosing appropriate imputation strategies\n",
    "2. Feature selection\n",
    "3. Understanding sensor reliability"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Function to calculate missing percentages\n",
    "def calculate_missing(df, dataset_name):\n",
    "    missing = df.isna().sum() / len(df) * 100\n",
    "    missing = missing[missing > 0].sort_values(ascending=False)\n",
    "    \n",
    "    print(f\"📊 Missing Value Analysis - {dataset_name}\")\n",
    "    print(\"=\"*60)\n",
    "    print(f\"Total columns with missing values: {len(missing)}\")\n",
    "    print(f\"Total missing values: {df.isna().sum().sum():,}\")\n",
    "    \n",
    "    # Create categories\n",
    "    categories = {\n",
    "        'High (80-100%)': len(missing[missing >= 80]),\n",
    "        'Medium (20-80%)': len(missing[(missing >= 20) & (missing < 80)]),\n",
    "        'Low (5-20%)': len(missing[(missing >= 5) & (missing < 20)]),\n",
    "        'Very Low (<5%)': len(missing[missing < 5])\n",
    "    }\n",
    "    \n",
    "    print(\"\\n📊 Missing Value Categories:\")\n",
    "    for category, count in categories.items():\n",
    "        print(f\"  {category}: {count} features\")\n",
    "    \n",
    "    return missing\n",
    "\n",
    "# Calculate missing for training and test\n",
    "train_missing = calculate_missing(train_df, 'Training Set')\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "test_missing = calculate_missing(test_df, 'Test Set')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize missing patterns\n",
    "fig = make_subplots(\n",
    "    rows=2, cols=1,\n",
    "    subplot_titles=('Training Set Missing Values', 'Test Set Missing Values'),\n",
    "    vertical_spacing=0.15\n",
    ")\n",
    "\n",
    "for idx, (missing_data, title) in enumerate([(train_missing, 'Training'), (test_missing, 'Test')], start=1):\n",
    "    if len(missing_data) > 0:\n",
    "        missing_df = pd.DataFrame({\n",
    "            'Feature': missing_data.index,\n",
    "            'Missing %': missing_data.values\n",
    "        }).head(30)  # Show top 30\n",
    "        \n",
    "        fig.add_trace(\n",
    "            go.Bar(\n",
    "                x=missing_df['Missing %'],\n",
    "                y=missing_df['Feature'],\n",
    "                orientation='h',\n",
    "                marker_color=COLORS['primary'],\n",
    "                text=missing_df['Missing %'].round(2),\n",
    "                textposition='outside'\n",
    "            ),\n",
    "            row=idx, col=1\n",
    "        )\n",
    "\n",
    "fig.update_layout(\n",
    "    height=800,\n",
    "    title_text='Top 30 Features with Missing Values',\n",
    "    showlegend=False,\n",
    "    plot_bgcolor='white',\n",
    "    font=dict(size=12)\n",
    ")\n",
    "fig.update_xaxes(title_text='Missing Percentage (%)', row=1, col=1)\n",
    "fig.update_xaxes(title_text='Missing Percentage (%)', row=2, col=1)\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 💼 Business Insight: Missing Values\n",
    "\n",
    "**Key Observations**:\n",
    "1. **Pattern of missingness**: Multiple features have high missing percentages\n",
    "2. **Sensor reliability**: Features with >80% missing may be from unreliable sensors or rarely triggered events\n",
    "3. **Data quality**: Missing values are consistent across train and test sets\n",
    "\n",
    "**Recommended Strategy**:\n",
    "1. **High missingness (>80%)**: Drop these features or create binary flags\n",
    "2. **Medium missingness (20-80%)**: Use advanced imputation (MICE, KNN)\n",
    "3. **Low missingness (<20%)**: Simple imputation (median/mode)\n",
    "4. **Create missing indicators**: Sometimes missingness itself is informative"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Feature Analysis\n",
    "\n",
    "Analyzing feature distributions helps us understand data patterns and identify potential issues."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select numeric features (excluding class)\n",
    "numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()\n",
    "numeric_cols = [col for col in numeric_cols if col != 'class']\n",
    "\n",
    "print(f\"📊 Number of numeric features: {len(numeric_cols)}\")\n",
    "print(f\"Sample features: {numeric_cols[:10]}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Feature statistics\n",
    "feature_stats = train_df[numeric_cols].describe().T\n",
    "feature_stats['skew'] = train_df[numeric_cols].skew()\n",
    "feature_stats['kurtosis'] = train_df[numeric_cols].kurtosis()\n",
    "feature_stats['missing_%'] = (train_df[numeric_cols].isna().sum() / len(train_df) * 100)\n",
    "\n",
    "# Display top features by missingness\n",
    "print(\"📊 TOP 10 FEATURES WITH HIGHEST MISSING %\")\n",
    "print(\"=\"*60)\n",
    "feature_stats_sorted = feature_stats.sort_values('missing_%', ascending=False)\n",
    "display(feature_stats_sorted.head(10)[['mean', 'std', 'missing_%']])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 📈 Feature Distribution Visualization\n",
    "\n",
    "Let's examine the distribution of key features by class to identify discriminative patterns."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Sample some features for visualization\n",
    "sample_features = ['aa_000', 'ab_000', 'ac_000', 'ad_000']\n",
    "\n",
    "fig = make_subplots(\n",
    "    rows=2, cols=2,\n",
    "    subplot_titles=[f'Distribution of {feat}' for feat in sample_features],\n",
    "    horizontal_spacing=0.1,\n",
    "    vertical_spacing=0.15\n",
    ")\n",
    "\n",
    "for idx, feature in enumerate(sample_features):\n",
    "    row = idx // 2 + 1\n",
    "    col = idx % 2 + 1\n",
    "    \n",
    "    # Get data for each class\n",
    "    for class_val, color in [('neg', COLORS['primary']), ('pos', COLORS['secondary'])]:\n",
    "        data = train_df[train_df['class'] == class_val][feature].dropna()\n",
    "        \n",
    "        fig.add_trace(\n",
    "            go.Histogram(\n",
    "                x=data,\n",
    "                name=f'{class_val.upper()}',\n",
    "                marker_color=color,\n",
    "                opacity=0.7,\n",
    "                nbinsx=30,\n",
    "                showlegend=(idx == 0)\n",
    "            ),\n",
    "            row=row, col=col\n",
    "        )\n",
    "\n",
    "fig.update_layout(\n",
    "    height=600,\n",
    "    title_text='Feature Distributions by Class',\n",
    "    barmode='overlay',\n",
    "    plot_bgcolor='white'\n",
    ")\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 💼 Business Insight: Feature Patterns\n",
    "\n",
    "**Key Observations**:\n",
    "1. **Different distributions**: Some features show clear separation between classes\n",
    "2. **Skewed distributions**: Many features are right-skewed\n",
    "3. **Zero-inflated**: Many features have large number of zeros\n",
    "\n",
    "**Implications**:\n",
    "1. **Feature transformation**: May need log transformation for skewed features\n",
    "2. **Zero handling**: Features with many zeros may indicate binary/sensor trigger events\n",
    "3. **Grouping features**: Histogram features (ay_000-ay_009) should be treated as distribution"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🔗 Correlation Analysis\n",
    "\n",
    "Understanding correlations helps identify:\n",
    "1. Multicollinearity (for linear models)\n",
    "2. Feature redundancy\n",
    "3. Important relationships with target"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create correlation matrix\n",
    "corr_matrix = train_df[numeric_cols].corr().abs()\n",
    "\n",
    "# Find highly correlated pairs\n",
    "upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))\n",
    "high_corr_pairs = [(col, row, upper.loc[col, row]) \n",
    "                   for col in upper.columns \n",
    "                   for row in upper.index \n",
    "                   if upper.loc[col, row] > 0.9]\n",
    "\n",
    "print(f\"📊 Found {len(high_corr_pairs)} highly correlated pairs (>0.9)\")\n",
    "if high_corr_pairs:\n",
    "    print(\"\\n📊 TOP 10 HIGHLY CORRELATED FEATURES:\")\n",
    "    for feat1, feat2, corr in sorted(high_corr_pairs, key=lambda x: x[2], reverse=True)[:10]:\n",
    "        print(f\"  {feat1} ↔ {feat2}: {corr:.3f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize correlation with target\n",
    "target_corr = train_df[numeric_cols].apply(lambda x: x.corr(train_df['class'].map({'neg': 0, 'pos': 1})))\n",
    "target_corr = target_corr.sort_values(ascending=False)\n",
    "\n",
    "fig = px.bar(\n",
    "    x=target_corr.values[:20],\n",
    "    y=target_corr.index[:20],\n",
    "    orientation='h',\n",
    "    title='Top 20 Features Correlated with Target (APS Failure)',\n",
    "    labels={'x': 'Correlation with Target', 'y': 'Feature'},\n",
    "    color=target_corr.values[:20],\n",
    "    color_continuous_scale=['blue', 'white', 'red']\n",
    ")\n",
    "\n",
    "fig.update_layout(\n",
    "    height=600,\n",
    "    plot_bgcolor='white',\n",
    "    font=dict(size=12)\n",
    ")\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 💼 Business Insight: Feature Importance\n",
    "\n",
    "**Key Observations**:\n",
    "1. **Strongest predictors**: Features like `aa_000`, `ab_000` show moderate correlation\n",
    "2. **Potential domain meaning**: These features likely represent critical APS components\n",
    "3. **Positive vs negative correlation**: Both directions indicate different failure mechanisms\n",
    "\n",
    "**Recommended Actions**:\n",
    "1. **Feature groups**: Group features by prefix (aa_, ab_, ay_, etc.)\n",
    "2. **Domain mapping**: Attempt to map features to APS components\n",
    "3. **Feature engineering**: Create composite features from highly correlated groups"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🎯 Feature Group Analysis: Histogram Features\n",
    "\n",
    "The dataset contains histogram features (ay_000-ay_009) which represent distributions. Let's analyze them."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Identify histogram features\n",
    "hist_features = [col for col in train_df.columns if col.startswith('ay_')]\n",
    "print(f\"📊 Histogram Features: {len(hist_features)}\")\n",
    "print(f\"   {hist_features}\")\n",
    "\n",
    "# Analyze histogram features by class\n",
    "hist_data = train_df.groupby('class')[hist_features].mean()\n",
    "\n",
    "fig = go.Figure()\n",
    "for class_val in ['neg', 'pos']:\n",
    "    fig.add_trace(go.Scatter(\n",
    "        x=hist_features,\n",
    "        y=hist_data.loc[class_val],\n",
    "        mode='lines+markers',\n",
    "        name=class_val.upper(),\n",
    "        line=dict(width=3),\n",
    "        marker=dict(size=10)\n",
    "    ))\n",
    "\n",
    "fig.update_layout(\n",
    "    title='Average Histogram Distribution by Class',\n",
    "    xaxis_title='Histogram Bins',\n",
    "    yaxis_title='Average Value',\n",
    "    plot_bgcolor='white',\n",
    "    height=500,\n",
    "    font=dict(size=14),\n",
    "    legend=dict(\n",
    "        x=0.02,\n",
    "        y=0.98,\n",
    "        bgcolor='rgba(255,255,255,0.9)',\n",
    "        bordercolor='rgba(0,0,0,0.1)',\n",
    "        borderwidth=1\n",
    "    )\n",
    ")\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 💼 Business Insight: Histogram Features\n",
    "\n",
    "**Key Observations**:\n",
    "1. **Different distributions**: APS failures (pos) show different histogram patterns\n",
    "2. **Shift in distribution**: The peak location differs between classes\n",
    "3. **Spread differences**: Variance across bins varies by class\n",
    "\n",
    "**Feature Engineering Opportunity**:\n",
    "1. **Statistical features**: Calculate mean, variance, skewness from histograms\n",
    "2. **Shape features**: Mode, entropy, peak location\n",
    "3. **Difference features**: Difference between consecutive bins\n",
    "\n",
    "**Business Interpretation**:\n",
    "- Histogram features likely represent pressure distribution over time\n",
    "- APS failures may show pressure instability or loss"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Summary Statistics\n",
    "\n",
    "### Data Quality Assessment"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Comprehensive summary\n",
    "print(\"=\"*80)\n",
    "print(\"📊 DATA QUALITY SUMMARY\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Size\n",
    "print(f\"\\n📁 Dataset Size:\")\n",
    "print(f\"  - Training: {train_df.shape[0]:,} samples\")\n",
    "print(f\"  - Test: {test_df.shape[0]:,} samples\")\n",
    "print(f\"  - Features: {train_df.shape[1] - 1}\")\n",
    "\n",
    "# Missing values\n",
    "missing_percent = (train_df.isna().sum().sum() / (train_df.shape[0] * train_df.shape[1]) * 100)\n",
    "print(f\"\\n🔍 Missing Values:\")\n",
    "print(f\"  - Overall missing: {missing_percent:.2f}%\")\n",
    "print(f\"  - Features with >50% missing: {len([col for col in train_df.columns if train_df[col].isna().sum() / len(train_df) > 0.5])}\")\n",
    "\n",
    "# Class balance\n",
    "print(f\"\\n⚖️ Class Balance:\")\n",
    "for class_val, count in class_counts.items():\n",
    "    print(f\"  - {class_val.upper()}: {count:,} ({count/len(train_df)*100:.2f}%)\")\n",
    "print(f\"  - Imbalance Ratio: {class_counts['neg']/class_counts['pos']:.2f}:1\")\n",
    "\n",
    "# Feature types\n",
    "hist_features = [col for col in train_df.columns if col.startswith('ay_')]\n",
    "counter_features = [col for col in train_df.columns if col.startswith('ag_')]\n",
    "print(f\"\\n🏷️ Feature Types:\")\n",
    "print(f\"  - Histogram features: {len(hist_features)}\")\n",
    "print(f\"  - Counter features: {len(counter_features)}\")\n",
    "print(f\"  - Binary/Indicator features: {len([col for col in train_df.columns if col.startswith(('bx_', 'by_', 'bz_', 'ca_'))])}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🎯 Summary & Next Steps\n",
    "\n",
    "### Key Findings\n",
    "\n",
    "1. **Class Imbalance**: Significant imbalance (neg:pos ratio ~ 100:1)\n",
    "2. **Missing Values**: Extensive missing data requiring careful handling\n",
    "3. **Feature Groups**: Clear grouping into counters, histograms, and indicators\n",
    "4. **Correlations**: Strong patterns within feature groups\n",
    "5. **Business Context**: Cost optimization is critical (FN: 500, FP: 10)\n",
    "\n",
    "### Recommended Next Steps\n",
    "\n",
    "#### 1. **Feature Engineering** (Next Notebook)\n",
    "   - Handle missing values strategically\n",
    "   - Engineer statistical features from histograms\n",
    "   - Create rate-of-change features from counters\n",
    "   - Build binary indicators for high-missing features\n",
    "\n",
    "#### 2. **Feature Selection**\n",
    "   - Remove features with >80% missing\n",
    "   - Select features with strong correlation to target\n",
    "   - Apply variance threshold\n",
    "\n",
    "#### 3. **Model Development**\n",
    "   - Use tree-based models (handle missing values well)\n",
    "   - Implement cost-sensitive learning\n",
    "   - Optimize threshold for business costs\n",
    "\n",
    "#### 4. **Evaluation Strategy**\n",
    "   - Focus on business cost minimization\n",
    "   - Use stratified cross-validation\n",
    "   - Monitor FN rates closely\n",
    "\n",
    "### 📋 Checklist for Phase 2\n",
    "\n",
    "- [ ] Handle `na` values systematically\n",
    "- [ ] Engineer histogram features (mean, variance, skewness)\n",
    "- [ ] Create rate-of-change features\n",
    "- [ ] Build missingness indicators\n",
    "- [ ] Apply log transformations to skewed features\n",
    "- [ ] Scale features (StandardScaler or RobustScaler)\n",
    "- [ ] Prepare data for modeling\n",
    "- [ ] Save processed data for Phase 3"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## 📝 Next: Feature Engineering & Preprocessing\n",
    "\n",
    "Proceed to `02_Feature_Engineering.ipynb` for comprehensive feature engineering and preprocessing pipeline.\n",
    "\n",
    "**Key Objectives**:\n",
    "1. Transform raw data into model-ready features\n",
    "2. Create domain-inspired features\n",
    "3. Optimize data quality and relevance\n",
    "4. Prepare for model training\n",
    "\n",
    "---\n",
    "\n",
    "*End of EDA Notebook*"
   ]
  }
 ]
}